### Importera verktyg, läsa in fil, konvertera till CSV

In [ ]:
#importerar in verktyg och bibliotek
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


#läser in vår fil
df_flanker = pd.read_excel("Raw_Flanker.xlsx")

#spara som csv i samma directory
df_flanker.to_csv("Raw_Flanker.csv", index=False)

### Utforska och få en överblick över datan
Titta efter NaN, se vilka kolumner vi behöver och hur vi på andra sätt kan förenkla vår data till vårt syfte

In [ ]:
#utforska datan

df_flanker = pd.read_csv("Raw_Flanker.csv")

print("Flanker info")
df_flanker.info()
display(df_flanker.head())


In [ ]:
#check for NaN

print("Tot NaN for Flanker-test")
df_flanker.isna().sum()



### Cleana vår data
Välja kolumner som vi vill ha kvar, sortera, skapa ny kolumn 'session', ta bort practiceruns, göra 'date' tydligare etc

In [ ]:
# läser in csv filen
df_flanker = pd.read_csv("Raw_Flanker.csv")

In [ ]:
# gör datumet mer lättläst
df_flanker['date'] = pd.to_datetime(df_flanker['date'].astype(str).str.zfill(6), format='%m%d%y', errors='coerce').dt.strftime('%Y-%m-%d')

In [ ]:
# sortera kronologiskt efter deltagare, datum och tid så sessionerna hamnar i rätt ordning
df_flanker = df_flanker.sort_values(by=['subjectid', 'date', 'time'])

In [ ]:
# skapa 'session'-kolumnen per deltagare (1 = tidigaste tillfället, 2 = nästa, 3 = sista)
df_flanker['session'] = df_flanker.groupby('subjectid')['date'].transform(lambda x: pd.factorize(x)[0] + 1)

In [ ]:
# ta bort övningsförsök
df_clean = df_flanker[df_flanker['values.practice'] == 0].copy()

In [ ]:
# välj enbart de kolumner vi vill behålla (inklusive 'session')
kolumner_att_behålla = ['date', 'time', 'subjectid', 'session', 'correct', 'values.congruence', 'latency']

In [ ]:
# kontrollera att kolumnerna finns
df_clean = df_clean[[col for col in kolumner_att_behålla if col in df_clean.columns]]

## Ta bort outliers

In [ ]:
# Exkludera personer med låg andel rätt 
ACC_GRANS = 0.80  # 80 %

acc_flanker = df_clean.groupby("subjectid")["correct"].mean()

låg = acc_flanker[acc_flanker < ACC_GRANS].index  # deltagare UNDER 80 % i Flanker

print(f"Deltagare före: {df_clean['subjectid'].nunique()}")
print(f"Exkluderas ({len(låg)} st):", list(låg))

df_clean = df_clean[~df_clean["subjectid"].isin(låg)].copy()  # behåller deltagare med ≥ 80 %

print(f"Deltagare efter: {df_clean['subjectid'].nunique()}")

In [ ]:
# ta bort outliers reaktionstid
#spara antalet rader innan vi tar bort gränsvärden
antal_fore = len(df_clean)

#sätta gränsvärden för outliers (200ms nedre gräns, 2000ms övre gräns)
df_clean = df_clean[(df_clean['latency'] >= 200) & (df_clean['latency'] <= 3000)].copy()

#ta reda på hur många vi tagit bort genom att jämföra före och efter
antal_efter = len(df_clean)
antal_borttagna = antal_fore - antal_efter

print(f"Antal borttagna försök: {antal_borttagna}")


### Kontrollera våran data
Se över att allt ser korrekt ut och som vi vill ha det, kan vi göra ytterligare förbättringar.

In [ ]:
#kontrollera våran data efter vi cleanat den

display(df_clean.head())
df_clean.info()

df_clean['subjectid'].nunique()

antal_unika = df_clean['subjectid'].nunique()
print(f"Antal unika deltagare: {antal_unika}")



In [ ]:
# skapa unika sessions med hjälpa av date och subject id
df_flanker['session'] = df_flanker.groupby('subjectid')['date'].transform(lambda x: pd.factorize(x)[0] + 1)

#kontrollera hur det ser ut för en person med 1 eller flera sessions
print(df_flanker[df_flanker['subjectid'] == 45][['subjectid', 'date', 'time', 'session']].drop_duplicates())

In [ ]:
# spara ner allt till EN gemensam CSV-fil (ta bort # för att spara ner en ny csv fil med cleanad data)

#df_clean.to_csv("flanker_all_sessions_combined.csv", index=False)

print("Ta bort # ovan för att spara ner en ny 'cleanad' csv, 'flanker_all_sessions_combined.csv'.")